# 00 · Probe the data stores
Checks every WeatherBench2 store the model uses: variables, leads, and how many 00 UTC inits per year.
Runs in ~5 min. **Settings: Internet ON, Accelerator None.** If a line says MISSING, stop and tell the team.

In [ ]:
# ── Setup (same in every notebook) ─────────────────────────────────────────────
# Kaggle: right panel → Settings → Internet ON.
import subprocess, sys, os
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gcsfs", "zarr>=2.18,<3"], check=True)

REPO = "/kaggle/working/repo"
BRANCH = "main"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH,
                    "https://github.com/shreyashsri79/081", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "-q"], check=True)
sys.path.insert(0, REPO)
COMMIT = subprocess.check_output(["git", "-C", REPO, "rev-parse", "--short", "HEAD"]).decode().strip()
print("code commit:", COMMIT)

import numpy as np, pandas as pd, xarray as xr
from blend import config as C
np.random.seed(C.SEED)
print("xarray", xr.__version__)

In [ ]:
from blend.sources import open_store, _lead_index

for model, stores in C.STORES.items():
    for url, years in stores:
        try:
            ds = open_store(url)
        except Exception as e:
            print(f"MISSING {model}: {url}\n   {e}")
            continue
        t = pd.DatetimeIndex(ds.time.values)
        per_year = pd.Series(t[t.hour == 0].year).value_counts().sort_index()
        per_year = {y: int(n) for y, n in per_year.items() if y in years}
        have = [v for v in C.VARS if v in ds] + (["(fuxi rain)"] if "total_precipitation_24hr_from_6hr" in ds else [])
        try:
            _lead_index(ds.prediction_timedelta); ok = "yes"
        except KeyError as e:
            ok = f"NO — {e}"
        print(f"{model:9s} {url.split('/')[-2]}/{url.split('/')[-1][:30]}")
        print(f"   vars: {have}")
        print(f"   00 UTC inits per year: {per_year}")
        print(f"   Day 1-10 leads present: {ok}")

In [ ]:
era5 = open_store(C.ERA5)
print("ERA5", str(era5.time.values[0])[:10], "->", str(era5.time.values[-1])[:10])
print("vars:", [v for v in C.VARS if v in era5])